# Práctica – Clase 5: Optimización II — técnicas avanzadas y regularización

En esta práctica trabajas dos ideas centrales de la clase: el momento como corrección al descenso de gradiente puro, y la elección del optimizador correcto para entrenar más rápido y generalizar mejor.

## Qué vas a practicar

- Implementas a mano, en NumPy, la regla de actualización del descenso de gradiente con momento y la aplicas sobre un valle estrecho, un paisaje de pérdida mal condicionado.
- Entrenas un perceptrón multicapa (MLP, por sus siglas en inglés) en PyTorch sobre un conjunto de datos sintético, comparando tres optimizadores — descenso de gradiente estocástico (SGD, por sus siglas en inglés) sin momento, SGD con momento y Adam — junto con dropout como técnica de regularización.

## Requisito previo

Antes de empezar, asegúrate de haber visto la Clase 5 en vivo (o de haber leído el capítulo "Optimización avanzada y regularización" del manual). Necesitas entender qué es el momento y por qué el número de condición de un problema afecta a la velocidad de convergencia.

Ejecuta este cuaderno en Google Colab. No necesitas instalar nada: todas las librerías que usas (NumPy, Matplotlib, PyTorch y scikit-learn) vienen preinstaladas.

## Ejercicio 1 — Descenso con momento a mano, sobre un valle estrecho

### Contexto

En clase viste que el descenso de gradiente puro avanza con lentitud en un valle estrecho: una dirección tiene mucha más curvatura que la otra, y el paso que evita divergir en la dirección empinada resulta demasiado pequeño para avanzar rápido en la dirección suave. El momento corrige esto acumulando una "velocidad" que suaviza esa lentitud y refuerza el avance consistente.

### Tarea

Completa la función `descenso_con_momento`, que implementa esta regla de actualización (la misma que usa PyTorch internamente):

```
v = beta * v + gradiente(punto_actual)
punto_actual = punto_actual - tasa_aprendizaje * v
```

Después, ejecuta la celda siguiente, que aplica tu función sobre una función de valle estrecho (una parábola con mucha más curvatura en un eje que en el otro) y compara la trayectoria con la del descenso de gradiente normal, ya implementado.

### Pistas

- `v` empieza en cero y tiene la misma forma que `punto_inicial`.
- En cada paso, primero actualizas `v` con el gradiente del punto actual y después usas ese `v` para mover el punto: no uses el gradiente directamente para mover el punto.
- Un valor típico de `beta` está entre 0,9 y 0,99: cuanto más alto, más memoria tiene la velocidad de los gradientes pasados.
- Guarda cada punto por el que pasas en la lista `trayectoria`, tal como ya hace `descenso_gradiente`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Valle estrecho: una parábola con mucha más curvatura en el eje x0 que en
# el eje x1. El cociente a / b es el número de condición del problema,
# igual que en la analogía del valle que viste en clase.
a, b = 100.0, 1.0


def perdida(punto):
    """Valor de la función de pérdida en un punto (x0, x1)."""
    x0, x1 = punto
    return 0.5 * (a * x0 ** 2 + b * x1 ** 2)


def gradiente(punto):
    """Gradiente de la función de pérdida en un punto (x0, x1)."""
    x0, x1 = punto
    return np.array([a * x0, b * x1])

In [ ]:
def descenso_gradiente(gradiente, punto_inicial, tasa_aprendizaje, n_pasos):
    """Descenso de gradiente normal. Devuelve la trayectoria completa,
    un array de forma (n_pasos + 1, 2), incluido el punto inicial."""
    punto_actual = np.array(punto_inicial, dtype=float)
    trayectoria = [punto_actual.copy()]

    for _ in range(n_pasos):
        punto_actual = punto_actual - tasa_aprendizaje * gradiente(punto_actual)
        trayectoria.append(punto_actual.copy())

    return np.array(trayectoria)

In [ ]:
def descenso_con_momento(gradiente, punto_inicial, tasa_aprendizaje, beta, n_pasos):
    """Descenso de gradiente con momento. Devuelve la trayectoria completa,
    un array de forma (n_pasos + 1, 2), incluido el punto inicial."""
    punto_actual = np.array(punto_inicial, dtype=float)
    v = np.zeros_like(punto_actual)
    trayectoria = [punto_actual.copy()]

    for _ in range(n_pasos):
        # TODO: actualiza la velocidad acumulada con el gradiente del punto actual
        v = ...

        # TODO: actualiza el punto usando la velocidad que acabas de calcular
        punto_actual = ...

        trayectoria.append(punto_actual.copy())

    return np.array(trayectoria)

In [ ]:
punto_inicial = [1.0, 1.0]
tasa_aprendizaje = 0.0095
n_pasos = 100

trayectoria_gd = descenso_gradiente(gradiente, punto_inicial, tasa_aprendizaje, n_pasos)
trayectoria_momento = descenso_con_momento(
    gradiente, punto_inicial, tasa_aprendizaje, beta=0.9, n_pasos=n_pasos
)

print(f"Pérdida final, GD normal: {perdida(trayectoria_gd[-1]):.6f}")
print(f"Pérdida final, GD con momento: {perdida(trayectoria_momento[-1]):.6f}")

perdidas_gd = [perdida(p) for p in trayectoria_gd]
perdidas_momento = [perdida(p) for p in trayectoria_momento]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: contorno de la pérdida con las dos trayectorias superpuestas
x0_rango = np.linspace(-1.2, 1.2, 200)
x1_rango = np.linspace(-1.2, 1.2, 200)
malla_x0, malla_x1 = np.meshgrid(x0_rango, x1_rango)
malla_z = 0.5 * (a * malla_x0 ** 2 + b * malla_x1 ** 2)

ax1.contour(malla_x0, malla_x1, malla_z, levels=30, cmap="Greys", alpha=0.6)
ax1.plot(trayectoria_gd[:, 0], trayectoria_gd[:, 1], "o-", markersize=3, label="GD normal")
ax1.plot(trayectoria_momento[:, 0], trayectoria_momento[:, 1], "o-", markersize=3, label="GD con momento")
ax1.set_xlabel("x0 (curvatura alta)")
ax1.set_ylabel("x1 (curvatura baja)")
ax1.set_title("Trayectorias sobre el valle estrecho")
ax1.legend()

# Panel derecho: pérdida frente a iteración, en escala logarítmica
ax2.plot(perdidas_gd, label="GD normal")
ax2.plot(perdidas_momento, label="GD con momento")
ax2.set_yscale("log")
ax2.set_xlabel("Iteración")
ax2.set_ylabel("Pérdida (escala log)")
ax2.set_title("Velocidad de convergencia")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<details>
<summary>💡 Ver solución</summary>

```python
def descenso_con_momento(gradiente, punto_inicial, tasa_aprendizaje, beta, n_pasos):
    """Descenso de gradiente con momento. Devuelve la trayectoria completa,
    un array de forma (n_pasos + 1, 2), incluido el punto inicial."""
    punto_actual = np.array(punto_inicial, dtype=float)
    v = np.zeros_like(punto_actual)
    trayectoria = [punto_actual.copy()]

    for _ in range(n_pasos):
        # La velocidad acumula el gradiente actual sobre la velocidad anterior,
        # con decaimiento exponencial controlado por beta.
        v = beta * v + gradiente(punto_actual)

        # El punto se mueve según la velocidad acumulada, no según el
        # gradiente instantáneo: así se suavizan los cambios de dirección.
        punto_actual = punto_actual - tasa_aprendizaje * v

        trayectoria.append(punto_actual.copy())

    return np.array(trayectoria)
```

Con `beta=0.9` y la misma tasa de aprendizaje que el descenso de gradiente normal, el momento llega a una pérdida final varios órdenes de magnitud menor en el mismo número de pasos: la velocidad acumulada compensa la lentitud en la dirección de curvatura baja sin que tengas que arriesgar una tasa de aprendizaje mayor.
</details>

## Ejercicio 2 — Elegir el mejor optimizador para entrenar en menos épocas

### Caso de uso

Imagina que colaboras en un proyecto de digitalización con una empresa de la cadena Agro-Mar Alimentación: necesitas entrenar un clasificador sencillo y el equipo técnico te pide que llegue a buena precisión en el menor número de épocas posible, porque el hardware disponible es limitado. Antes de complicar la arquitectura, la primera palanca que tienes es elegir bien el optimizador.

### Tarea

Entrenas el mismo perceptrón multicapa (MLP) sobre un conjunto de datos sintético no lineal (`make_moons`, de scikit-learn) con tres optimizadores distintos: SGD sin momento, SGD con momento y Adam. El modelo incluye una capa `torch.nn.Dropout` para regularizar, tal como viste en la parte final de la clase.

Completa la instanciación de cada optimizador y define un valor de `weight_decay` (decaimiento de pesos) razonable. Al final, superpones las tres curvas de pérdida para comparar qué optimizador converge más rápido con el mismo número de épocas.

### Pistas

- `torch.optim.SGD` acepta los parámetros `lr`, `momentum` y `weight_decay`.
- `torch.optim.Adam` acepta `lr` y `weight_decay`; no necesita `momentum` porque ya incorpora una lógica de acumulación equivalente.
- Un valor típico de `weight_decay` está entre 1e-5 y 1e-3: cuanto más alto, más penalizas los pesos grandes.
- Recuerda que `modelo.train()` activa el dropout durante el entrenamiento, ya incluido en la función `entrenar`.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.manual_seed(0)

# --- Datos sintéticos no lineales ---
X_np, y_np = make_moons(n_samples=500, noise=0.25, random_state=0)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.long)

plt.figure(figsize=(5, 4))
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap="coolwarm", s=15)
plt.title("Conjunto de datos make_moons")
plt.show()

In [ ]:
class MLP(nn.Module):
    """Perceptrón multicapa (MLP) pequeño con dropout para regularizar."""

    def __init__(self, prob_dropout=0.3):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Dropout(prob_dropout),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(prob_dropout),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.capas(x)


def crear_modelo():
    """Crea el modelo con los mismos pesos iniciales en cada prueba,
    para que la comparación entre optimizadores sea justa."""
    torch.manual_seed(42)
    return MLP()

In [ ]:
def entrenar(optimizador_fn, n_epocas=150):
    modelo = crear_modelo()
    perdida_fn = nn.CrossEntropyLoss()
    optimizador = optimizador_fn(modelo.parameters())

    historial_perdida = []
    for _ in range(n_epocas):
        modelo.train()  # activa el dropout durante el entrenamiento
        optimizador.zero_grad()
        pred = modelo(X)
        perdida = perdida_fn(pred, y)
        perdida.backward()
        optimizador.step()
        historial_perdida.append(perdida.item())

    return historial_perdida

In [ ]:
tasa_aprendizaje = 0.05
decaimiento_pesos = ...  # TODO: define un valor típico de weight_decay (entre 1e-5 y 1e-3)

resultados = {
    "SGD (sin momento)": entrenar(
        lambda params: ...  # TODO: instancia torch.optim.SGD sin momento, con lr=tasa_aprendizaje y weight_decay=decaimiento_pesos
    ),
    "SGD con momento (0.9)": entrenar(
        lambda params: ...  # TODO: instancia torch.optim.SGD con momentum=0.9, lr=tasa_aprendizaje y weight_decay=decaimiento_pesos
    ),
    "Adam": entrenar(
        lambda params: ...  # TODO: instancia torch.optim.Adam con lr=tasa_aprendizaje y weight_decay=decaimiento_pesos
    ),
}

In [ ]:
plt.figure(figsize=(8, 5))
for nombre, perdidas in resultados.items():
    plt.plot(perdidas, label=nombre)

plt.xlabel("Época")
plt.ylabel("Pérdida (CrossEntropy)")
plt.title("Comparación de optimizadores en make_moons")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

<details>
<summary>💡 Ver solución</summary>

```python
tasa_aprendizaje = 0.05
decaimiento_pesos = 1e-4  # valor típico: penaliza levemente los pesos grandes

resultados = {
    "SGD (sin momento)": entrenar(
        lambda params: torch.optim.SGD(
            params, lr=tasa_aprendizaje, weight_decay=decaimiento_pesos
        )
    ),
    "SGD con momento (0.9)": entrenar(
        lambda params: torch.optim.SGD(
            params, lr=tasa_aprendizaje, momentum=0.9, weight_decay=decaimiento_pesos
        )
    ),
    "Adam": entrenar(
        lambda params: torch.optim.Adam(
            params, lr=tasa_aprendizaje, weight_decay=decaimiento_pesos
        )
    ),
}
```

Observa el resultado: SGD sin momento es, con diferencia, el más lento en bajar la pérdida en las primeras épocas. SGD con momento acelera esa caída al suavizar la dirección de descenso. Adam suele converger todavía más rápido al principio, porque adapta la tasa de aprendizaje por parámetro además de acumular momento, tal como viste en la clase.
</details>

## Para cerrar

Ya has visto con tus propios ojos por qué el momento no es un detalle técnico menor: en el mismo valle estrecho, la diferencia entre usarlo o no se mide en órdenes de magnitud de pérdida final. Y en un problema de clasificación real, elegir bien entre SGD, SGD con momento y Adam, junto con dropout y weight decay, es la diferencia entre esperar horas o minutos por un modelo entrenado.

**Vuelve ahora a tu propio proyecto, cambia el optimizador de tu modelo actual y comprueba cuántas épocas necesitas para llegar a la misma pérdida.**